## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [ ]:
import pandas as pd
import seaborn as sns


titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [ ]:
missing_in_columns = titanic_data.isnull().sum()
missing_in_columns

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [ ]:
rows = titanic_data.shape[0]
titanic_data = titanic_data.dropna(thresh=rows // 2, axis="columns")

cols = titanic_data.shape[1]
titanic_data = titanic_data.dropna(thresh=cols // 2, axis="rows")

titanic_data.isnull().sum()

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [ ]:
for who in ['man', 'woman', 'child']:
    median_age = titanic_data[titanic_data['who'] == who]['age'].median().round()
    print(median_age)
    who_mask = titanic_data['who'] == who
    titanic_data.loc[who_mask, 'age'] = titanic_data.loc[who_mask, 'age'].fillna(median_age)

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [ ]:
titanic_data = titanic_data.dropna(thresh=titanic_data.shape[1] - 1, axis="rows")
titanic_data.isnull().sum()

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [ ]:
print(titanic_data['embark_town'].value_counts().idxmax())

### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [ ]:
print(round(titanic_data['survived'].mean() * 100, 2))

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [ ]:
titanic_data[titanic_data['survived'] == True]['embark_town'].value_counts()

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [ ]:
percents = {}

for cl in titanic_data['class'].unique():
    class_mask = titanic_data['class'] == cl
    percents[cl] = round(titanic_data[class_mask]['survived'].mean() * 100, 2)

percents = pd.Series(percents)
percents

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [ ]:
rich_mask = titanic_data['fare'] >= 100
print(round(titanic_data[rich_mask]['survived'].mean() * 100, 2))

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [ ]:
lonely_child_mask = (titanic_data['who'] == 'child') & titanic_data['alone']
print(len(titanic_data[lonely_child_mask]))

Какие выводы вы можете сделать о выживших пассажирах Титаника?

Надо делать домашние задания по питону, закончить Физтех, стать миллионером и покупать билеты на Титаник только в первый класс, вероятность выжить в 2,5 раза больше! А богатым женщинам вообще больше всего повезло получается, делайте выводы...